# 20 — Advanced PyTorch Training Techniques

In the previous notebook, we learned transfer learning and fine-tuning with pretrained models.

Now we will study techniques that make real PyTorch training loops more flexible, efficient, and robust.

These techniques become especially useful when:

- Models are larger
- GPU memory is limited
- Training takes many epochs
- Learning rates need to change during training
- You want to resume interrupted experiments
- You need faster inference
- You want better instrumentation and debugging

## In this notebook, we will study:

1. Learning-rate schedulers
2. `StepLR`
3. `ReduceLROnPlateau`
4. Cosine annealing
5. Warmup intuition
6. Mixed-precision training
7. `torch.autocast`
8. Gradient scaling
9. Gradient accumulation
10. Larger effective batch sizes
11. Gradient clipping in real loops
12. Efficient inference
13. `torch.inference_mode()`
14. Checkpoint resuming
15. Training-loop instrumentation
16. Practical performance optimization
17. Common advanced-training mistakes
18. Practice exercises

## Main Goal

By the end of this notebook, you should understand how a basic training loop evolves into a more practical training system:

$$
\boxed{
\text{Batch}
\rightarrow
\text{Autocast}
\rightarrow
\text{Loss}
\rightarrow
\text{Scaled Backward}
\rightarrow
\text{Clip}
\rightarrow
\text{Optimizer}
\rightarrow
\text{Scheduler}
}
$$

while still preserving the core PyTorch principles you already learned.


In [ ]:
import copy
import json
import math
import time
from pathlib import Path

import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

print("PyTorch version:", torch.__version__)


# 1. Why Advanced Training Techniques Matter

A simple training loop is:

```python
optimizer.zero_grad()
outputs = model(inputs)
loss = criterion(outputs, targets)
loss.backward()
optimizer.step()
```

This is still the foundation.

Advanced techniques do not replace it.

They modify specific parts of the loop to improve:

- Optimization
- Numerical efficiency
- Memory usage
- Stability
- Recoverability
- Monitoring


# 2. Our Small Demonstration Problem

We will create a simple multi-class classification problem.

Input:

$$
20
$$

features.

Classes:

$$
3
$$

The dataset is intentionally simple so we can focus on training mechanics.


In [ ]:
torch.manual_seed(42)

num_samples = 1200
input_dim = 20
num_classes = 3

features = torch.randn(
    num_samples,
    input_dim
)

true_weights = torch.randn(
    input_dim,
    num_classes
)

scores = features @ true_weights

targets = scores.argmax(
    dim=1
)

print("Features:", features.shape)
print("Targets:", targets.shape)


# 3. Train / Validation Split


In [ ]:
train_size = 900

train_features = features[
    :train_size
]

train_targets = targets[
    :train_size
]

val_features = features[
    train_size:
]

val_targets = targets[
    train_size:
]

train_dataset = TensorDataset(
    train_features,
    train_targets
)

val_dataset = TensorDataset(
    val_features,
    val_targets
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)


# 4. Baseline Model

We will use a small MLP:

$$
20
\rightarrow
64
\rightarrow
64
\rightarrow
3
$$


In [ ]:
class SmallMLP(nn.Module):
    def __init__(
        self,
        input_dim=20,
        num_classes=3
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                64
            ),
            nn.ReLU(),

            nn.Linear(
                64,
                64
            ),
            nn.ReLU(),

            nn.Linear(
                64,
                num_classes
            )
        )

    def forward(self, x):
        return self.network(x)

model = SmallMLP()

print(model)


# 5. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# 6. Learning Rate Is Not Required to Stay Constant

So far, we often used one fixed learning rate:

$$
\eta=0.001
$$

for the entire training run.

But different phases of training may benefit from different learning rates.

Early training may need larger steps.

Later training may benefit from smaller steps.

This motivates:

> **Learning-rate schedulers**


# 7. What Is a Learning-Rate Scheduler?

A scheduler changes the optimizer's learning rate during training.

Conceptually:

$$
\boxed{
\eta_1
\rightarrow
\eta_2
\rightarrow
\eta_3
\rightarrow
\cdots
}
$$

The optimizer still performs the parameter update.

The scheduler changes the learning rate used by that optimizer.


# 8. Inspecting the Current Learning Rate

The optimizer stores learning rates inside:

```python
optimizer.param_groups
```


In [ ]:
model = SmallMLP()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

current_lr = optimizer.param_groups[
    0
]["lr"]

print(
    "Current learning rate:",
    current_lr
)


# 9. `StepLR`

`StepLR` multiplies the learning rate by a factor after a fixed number of epochs.

Example:

```python
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)
```

means:

> Every 5 scheduler steps, multiply learning rate by 0.1.


In [ ]:
model = SmallMLP()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

scheduler = (
    torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=5,
        gamma=0.1
    )
)

for epoch in range(12):
    lr = optimizer.param_groups[
        0
    ]["lr"]

    print(
        f"Epoch {epoch + 1:02d} | "
        f"lr = {lr:.5f}"
    )

    optimizer.step()
    scheduler.step()


# 10. Understanding the `StepLR` Pattern

With:

$$
step\_size=5
$$

and:

$$
gamma=0.1
$$

the learning rate changes approximately like:

$$
0.1
\rightarrow
0.01
\rightarrow
0.001
$$

This is a simple piecewise schedule.


# 11. Where Should `scheduler.step()` Go?

For many epoch-based schedulers, a common pattern is:

```python
for epoch in range(epochs):
    train_one_epoch(...)
    validate(...)
    scheduler.step()
```

So the scheduler advances once per epoch.

But not every scheduler follows exactly the same rule.

Always check what quantity the scheduler expects.


# 12. A Training Loop With `StepLR`


In [ ]:
def train_epoch_basic(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, targets in loader:
        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            inputs
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

    return (
        total_loss
        / total_samples
    )


In [ ]:
def evaluate_loss(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                inputs
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = targets.size(0)

            total_loss += (
                loss.item()
                * batch_size
            )

            total_samples += batch_size

    return (
        total_loss
        / total_samples
    )


In [ ]:
torch.manual_seed(42)

step_model = SmallMLP().to(
    device
)

criterion = nn.CrossEntropyLoss()

step_optimizer = torch.optim.SGD(
    step_model.parameters(),
    lr=0.1
)

step_scheduler = (
    torch.optim.lr_scheduler.StepLR(
        step_optimizer,
        step_size=3,
        gamma=0.5
    )
)

for epoch in range(8):
    train_loss = train_epoch_basic(
        step_model,
        train_loader,
        criterion,
        step_optimizer,
        device
    )

    val_loss = evaluate_loss(
        step_model,
        val_loader,
        criterion,
        device
    )

    current_lr = (
        step_optimizer
        .param_groups[0]["lr"]
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"LR {current_lr:.5f}"
    )

    step_scheduler.step()


# 13. `ReduceLROnPlateau`

`ReduceLROnPlateau` is different.

Instead of changing the learning rate after a fixed number of epochs, it watches a metric.

For example:

> Reduce the learning rate when validation loss stops improving.


In [ ]:
model = SmallMLP()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

plateau_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )
)

print(
    plateau_scheduler
)


# 14. `ReduceLROnPlateau` Needs a Metric

Unlike `StepLR`, we call:

```python
scheduler.step(
    validation_loss
)
```

because the scheduler needs to know whether the monitored metric improved.


In [ ]:
example_val_losses = [
    0.8,
    0.6,
    0.5,
    0.51,
    0.52,
    0.53,
    0.48
]

model = SmallMLP()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=1
    )
)

for epoch, val_loss in enumerate(
    example_val_losses,
    start=1
):
    scheduler.step(
        val_loss
    )

    print(
        f"Epoch {epoch} | "
        f"Val loss {val_loss:.3f} | "
        f"LR {optimizer.param_groups[0]['lr']:.6f}"
    )


# 15. `StepLR` vs `ReduceLROnPlateau`

$$
\begin{array}{|c|c|}
\hline
\textbf{StepLR} & \textbf{ReduceLROnPlateau} \\
\hline
Changes\ on\ fixed\ schedule & Reacts\ to\ metric \\
\hline
Does\ not\ inspect\ validation & Usually\ uses\ validation\ metric \\
\hline
scheduler.step() & scheduler.step(metric) \\
\hline
\end{array}
$$


# 16. Cosine Annealing

Cosine annealing gradually changes the learning rate following a cosine-shaped curve.

A simplified intuition is:

$$
\boxed{
\text{Large LR}
\rightarrow
\text{Smoothly Smaller LR}
}
$$

PyTorch provides:

```python
torch.optim.lr_scheduler.CosineAnnealingLR
```


In [ ]:
model = SmallMLP()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

cosine_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=10,
        eta_min=0.001
    )
)

for epoch in range(10):
    print(
        f"Epoch {epoch + 1:02d} | "
        f"LR {optimizer.param_groups[0]['lr']:.6f}"
    )

    optimizer.step()
    cosine_scheduler.step()


# 17. Cosine Annealing Intuition

A cosine schedule avoids abrupt learning-rate drops.

It can be useful when training over a known number of epochs.

The learning rate decreases smoothly toward:

$$
eta\_min
$$

over the scheduler cycle.


# 18. Warmup Intuition

Warmup begins training with a smaller learning rate and gradually increases it toward the target learning rate.

Conceptually:

$$
\boxed{
\text{Small LR}
\rightarrow
\text{Target LR}
\rightarrow
\text{Main Schedule}
}
$$

Warmup can help stabilize the earliest training steps, especially in large models or large-batch training.


# 19. Simple Linear Warmup Formula

If warmup lasts:

$$
W
$$

steps, a simple warmup factor is:

$$
\boxed{
factor(t)
=
\frac{t+1}{W}
}
$$

for early steps.

Then the learning rate gradually approaches its full value.


In [ ]:
def warmup_factor(
    step,
    warmup_steps
):
    if step >= warmup_steps:
        return 1.0

    return (
        step + 1
    ) / warmup_steps

for step in range(8):
    print(
        step,
        warmup_factor(
            step,
            warmup_steps=5
        )
    )


# 20. Warmup With `LambdaLR`

One way to implement simple warmup is with:

`LambdaLR`.


In [ ]:
model = SmallMLP()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

warmup_steps = 5

scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: min(
            1.0,
            (step + 1)
            / warmup_steps
        )
    )
)

for step in range(8):
    print(
        f"Step {step:02d} | "
        f"LR {optimizer.param_groups[0]['lr']:.6f}"
    )

    optimizer.step()
    scheduler.step()


# 21. Scheduler Granularity Matters

Some schedulers advance:

- Once per epoch

Others may advance:

- Once per optimizer update

Warmup is often step-based.

So always know whether your scheduler counts:

$$
epochs
$$

or:

$$
optimization\ steps
$$


# 22. Mixed-Precision Training

Modern GPUs can perform some operations faster using lower-precision floating-point formats.

Mixed precision combines:

- Lower precision where safe
- Higher precision where needed

The goal is often:

- Faster training
- Lower GPU memory usage

while preserving useful numerical stability.


# 23. Full Precision vs Mixed Precision

Traditional training commonly uses:

`float32`

Mixed precision may use lower precision such as:

- `float16`
- `bfloat16`

for selected operations.

Not every operation should necessarily run at the same dtype.


# 24. `torch.autocast`

PyTorch provides automatic mixed precision through:

```python
torch.autocast(...)
```

Inside the context, PyTorch chooses appropriate dtypes for supported operations.


In [ ]:
amp_enabled = (
    device.type == "cuda"
)

autocast_dtype = (
    torch.float16
    if device.type == "cuda"
    else torch.bfloat16
)

print(
    "AMP enabled:",
    amp_enabled
)

print(
    "Autocast dtype:",
    autocast_dtype
)


# 25. Basic `torch.autocast` Example

On CUDA, we can write:

```python
with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    outputs = model(inputs)
    loss = criterion(outputs, targets)
```

In this notebook, we enable AMP only when CUDA is available.


In [ ]:
amp_model = SmallMLP().to(
    device
)

inputs = torch.randn(
    16,
    20,
    device=device
)

targets = torch.randint(
    0,
    3,
    (16,),
    device=device
)

criterion = nn.CrossEntropyLoss()

with torch.autocast(
    device_type=device.type,
    dtype=autocast_dtype,
    enabled=amp_enabled
):
    outputs = amp_model(
        inputs
    )

    loss = criterion(
        outputs,
        targets
    )

print(
    "Output dtype:",
    outputs.dtype
)

print(
    "Loss dtype:",
    loss.dtype
)


# 26. Why Gradient Scaling Is Needed

With `float16`, very small gradients can underflow toward zero.

Gradient scaling temporarily multiplies the loss by a large scale before backward.

Conceptually:

$$
L_{scaled}
=
sL
$$

Then:

$$
\nabla(sL)
=
s\nabla L
$$

The optimizer later uses unscaled gradients.

This helps preserve small gradient values during low-precision training.


# 27. `GradScaler`

For CUDA mixed-precision training, PyTorch provides a gradient scaler.

A common modern pattern is:

```python
scaler = torch.amp.GradScaler(
    "cuda"
)
```

We can disable it automatically when CUDA is unavailable.


In [ ]:
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)

print(
    scaler
)


# 28. Mixed-Precision Training Step

The usual order becomes:

```python
optimizer.zero_grad()

with torch.autocast(...):
    outputs = model(inputs)
    loss = criterion(outputs, targets)

scaler.scale(loss).backward()

scaler.step(optimizer)

scaler.update()
```


In [ ]:
amp_model = SmallMLP().to(
    device
)

optimizer = torch.optim.AdamW(
    amp_model.parameters(),
    lr=1e-3
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)

inputs, targets = next(
    iter(train_loader)
)

inputs = inputs.to(
    device
)

targets = targets.to(
    device
)

optimizer.zero_grad(
    set_to_none=True
)

with torch.autocast(
    device_type=device.type,
    dtype=autocast_dtype,
    enabled=amp_enabled
):
    logits = amp_model(
        inputs
    )

    loss = criterion(
        logits,
        targets
    )

scaler.scale(
    loss
).backward()

scaler.step(
    optimizer
)

scaler.update()

print(
    "Loss:",
    loss.item()
)


# 29. Why `scaler.step()` Instead of `optimizer.step()`?

When using gradient scaling, the scaler needs to:

- Unscale/check gradients
- Detect invalid values
- Decide whether the optimizer step is safe

So the scaler wraps the optimizer step.


# 30. Mixed Precision Is Mainly a GPU Optimization

AMP can provide the biggest benefit on compatible accelerators.

On CPU:

- Behavior differs
- Speedups may be limited
- `bfloat16` support depends on hardware/operations

Do not assume mixed precision is automatically faster everywhere.


# 31. Gradient Clipping With AMP

When using a scaler, gradients are initially scaled.

If you want to clip the **true** gradients, first unscale them:

```python
scaler.unscale_(optimizer)
```

Then clip.


In [ ]:
amp_model = SmallMLP().to(
    device
)

optimizer = torch.optim.AdamW(
    amp_model.parameters(),
    lr=1e-3
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)

inputs, targets = next(
    iter(train_loader)
)

inputs = inputs.to(
    device
)

targets = targets.to(
    device
)

optimizer.zero_grad(
    set_to_none=True
)

with torch.autocast(
    device_type=device.type,
    dtype=autocast_dtype,
    enabled=amp_enabled
):
    logits = amp_model(
        inputs
    )

    loss = criterion(
        logits,
        targets
    )

scaler.scale(
    loss
).backward()

scaler.unscale_(
    optimizer
)

grad_norm = (
    torch.nn.utils.clip_grad_norm_(
        amp_model.parameters(),
        max_norm=1.0
    )
)

scaler.step(
    optimizer
)

scaler.update()

print(
    "Gradient norm before clipping:",
    grad_norm.item()
)


# 32. Correct AMP + Gradient-Clipping Order

Remember:

$$
\boxed{
zero\_grad
\rightarrow
autocast\ forward
\rightarrow
scaled\ backward
\rightarrow
unscale
\rightarrow
clip
\rightarrow
scaler.step
\rightarrow
scaler.update
}
$$


# 33. Gradient Accumulation

Sometimes your desired batch size does not fit in GPU memory.

Suppose you can only process:

$$
batch\_size=16
$$

but want an effective batch size of:

$$
64
$$

You can accumulate gradients across:

$$
4
$$

mini-batches before taking one optimizer step.


# 34. Effective Batch Size

Approximately:

$$
\boxed{
effective\ batch
=
micro\ batch
\times
accumulation\ steps
}
$$

Example:

$$
16\times4=64
$$


In [ ]:
micro_batch_size = 16
accumulation_steps = 4

effective_batch_size = (
    micro_batch_size
    * accumulation_steps
)

print(
    "Effective batch size:",
    effective_batch_size
)


# 35. Why Divide the Loss During Accumulation?

If you accumulate:

$$
K
$$

mini-batches, each backward pass adds gradients.

To approximate the average gradient over the larger effective batch, divide each mini-batch loss by:

$$
K
$$

before backward:

```python
loss = loss / accumulation_steps
```


# 36. Basic Gradient-Accumulation Loop


In [ ]:
accum_model = SmallMLP().to(
    device
)

optimizer = torch.optim.Adam(
    accum_model.parameters(),
    lr=1e-3
)

criterion = nn.CrossEntropyLoss()

accumulation_steps = 4

optimizer.zero_grad()

for batch_index, (
    inputs,
    targets
) in enumerate(
    train_loader
):
    inputs = inputs.to(
        device
    )

    targets = targets.to(
        device
    )

    logits = accum_model(
        inputs
    )

    loss = criterion(
        logits,
        targets
    )

    loss = (
        loss
        / accumulation_steps
    )

    loss.backward()

    should_step = (
        (batch_index + 1)
        % accumulation_steps
        == 0
    )

    is_last_batch = (
        batch_index + 1
        == len(train_loader)
    )

    if should_step or is_last_batch:
        optimizer.step()
        optimizer.zero_grad()

print(
    "Gradient accumulation pass complete."
)


# 37. Gradient Accumulation Is Not Exactly the Same in Every Situation

The effective batch may be similar, but exact training behavior can still differ because of:

- BatchNorm
- Dropout
- Data augmentation
- Optimizer state timing
- Scheduler timing

So think of accumulation as a practical approximation to larger-batch optimization.


# 38. BatchNorm and Gradient Accumulation

BatchNorm statistics are still computed on the **micro-batch**, not the effective accumulated batch.

Example:

$$
micro\ batch=16
$$

$$
accumulation\ steps=4
$$

The optimizer may behave like a 64-sample update, but BatchNorm still sees groups of 16.

This can matter for small micro-batches.


# 39. Gradient Accumulation With Mixed Precision

We can combine:

- Autocast
- GradScaler
- Gradient accumulation

The key rule remains:

> Scale the loss appropriately before backward.


In [ ]:
def train_one_epoch_advanced(
    model,
    loader,
    criterion,
    optimizer,
    device,
    accumulation_steps=1,
    max_grad_norm=None,
    use_amp=True
):
    model.train()

    use_amp = (
        use_amp
        and device.type == "cuda"
    )

    dtype = (
        torch.float16
        if device.type == "cuda"
        else torch.bfloat16
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    total_loss = 0.0
    total_samples = 0

    for batch_index, (
        inputs,
        targets
    ) in enumerate(
        loader
    ):
        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        with torch.autocast(
            device_type=device.type,
            dtype=dtype,
            enabled=use_amp
        ):
            logits = model(
                inputs
            )

            raw_loss = criterion(
                logits,
                targets
            )

            loss = (
                raw_loss
                / accumulation_steps
            )

        scaler.scale(
            loss
        ).backward()

        should_step = (
            (batch_index + 1)
            % accumulation_steps
            == 0
        )

        is_last_batch = (
            batch_index + 1
            == len(loader)
        )

        if should_step or is_last_batch:
            if max_grad_norm is not None:
                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=max_grad_norm
                )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        batch_size = targets.size(0)

        total_loss += (
            raw_loss.item()
            * batch_size
        )

        total_samples += (
            batch_size
        )

    return (
        total_loss
        / total_samples
    )


# 40. Testing the Advanced Training Function


In [ ]:
torch.manual_seed(42)

advanced_model = SmallMLP().to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    advanced_model.parameters(),
    lr=1e-3
)

train_loss = train_one_epoch_advanced(
    advanced_model,
    train_loader,
    criterion,
    optimizer,
    device,
    accumulation_steps=2,
    max_grad_norm=1.0,
    use_amp=True
)

print(
    "Training loss:",
    train_loss
)


# 41. Gradient Clipping in Real Training Loops

Gradient clipping is especially useful when:

- Gradients occasionally spike
- Recurrent or very deep models are unstable
- Fine-tuning is sensitive
- Large learning rates cause sudden updates

The common norm-based pattern is:

```python
clip_grad_norm_(
    model.parameters(),
    max_norm=...
)
```


# 42. Clipping Is a Safety Mechanism, Not a Substitute for Debugging

If gradients are constantly huge, also inspect:

- Learning rate
- Initialization
- Input normalization
- Loss formulation
- Model architecture

Gradient clipping can reduce the damage from spikes, but persistent exploding gradients still deserve investigation.


# 43. `optimizer.zero_grad(set_to_none=True)`

Instead of filling gradients with zeros:

```python
optimizer.zero_grad()
```

you can use:

```python
optimizer.zero_grad(
    set_to_none=True
)
```

This can reduce memory writes and may improve performance.

It also makes unused gradients remain:

`None`

which can be useful for debugging.


In [ ]:
model = SmallMLP()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

optimizer.zero_grad(
    set_to_none=True
)

for name, parameter in (
    model.named_parameters()
):
    print(
        name,
        parameter.grad
    )


# 44. Efficient Inference

Inference differs from training.

During inference we usually want:

- `model.eval()`
- No gradient tracking
- Batched predictions
- Efficient device transfer
- No optimizer
- No backward pass


# 45. `torch.no_grad()` vs `torch.inference_mode()`

`torch.inference_mode()` is a stronger inference-only context.

It disables gradient tracking and can reduce additional Autograd overhead.

Typical inference:

```python
model.eval()

with torch.inference_mode():
    outputs = model(inputs)
```


In [ ]:
inference_model = SmallMLP().to(
    device
)

inference_model.eval()

inputs = torch.randn(
    64,
    20
).to(
    device
)

with torch.inference_mode():
    outputs = inference_model(
        inputs
    )

print(
    "Output:",
    outputs.shape
)

print(
    "Requires grad:",
    outputs.requires_grad
)


# 46. When to Use `inference_mode()`

Use it when you are purely doing inference/evaluation and do not need Autograd-related features.

For most validation and deployment-style prediction loops, it is a strong default.

If you specifically need some Autograd-sensitive tensor behavior, use `no_grad()` instead.


# 47. Efficient Batched Inference

Do not necessarily run one sample at a time.

Batching can improve hardware utilization.

Example:

$$
1
\ sample
\rightarrow
many\ small\ kernel\ launches
$$

versus:

$$
64\ samples
\rightarrow
one\ larger\ batch
$$


In [ ]:
def predict_loader(
    model,
    loader,
    device
):
    model.eval()

    all_predictions = []

    with torch.inference_mode():
        for inputs, _ in loader:
            inputs = inputs.to(
                device
            )

            logits = model(
                inputs
            )

            predictions = logits.argmax(
                dim=1
            )

            all_predictions.append(
                predictions.cpu()
            )

    return torch.cat(
        all_predictions
    )


# 48. Inference Batch Size

Inference batch size can often be larger than training batch size because inference does not store backward graphs.

But the best value still depends on:

- GPU memory
- Input size
- Model size
- Latency requirements


# 49. DataLoader Performance

Data loading can become a bottleneck.

Important settings include:

- `batch_size`
- `num_workers`
- `pin_memory`
- `persistent_workers`

The best settings depend on your system.

Benchmark instead of assuming.


# 50. `pin_memory`

When training on CUDA, pinned CPU memory can help CPU-to-GPU transfers.

A common pattern:

```python
pin_memory=True
```

when using a CUDA device.


In [ ]:
pin_memory = (
    device.type == "cuda"
)

performance_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory
)

print(
    "pin_memory:",
    performance_loader.pin_memory
)


# 51. `non_blocking=True`

When tensors come from pinned CPU memory, CUDA transfers can potentially overlap more efficiently when using:

```python
tensor.to(
    device,
    non_blocking=True
)
```

This mainly matters in CUDA pipelines.


In [ ]:
inputs, targets = next(
    iter(performance_loader)
)

inputs = inputs.to(
    device,
    non_blocking=(
        device.type == "cuda"
    )
)

targets = targets.to(
    device,
    non_blocking=(
        device.type == "cuda"
    )
)

print(
    inputs.device,
    targets.device
)


# 52. `num_workers`

Increasing `num_workers` can allow data loading and preprocessing to happen in parallel.

But too many workers may:

- Increase RAM use
- Add process overhead
- Slow notebooks
- Cause platform-specific issues

Start with:

```python
num_workers=0
```

for debugging, then benchmark.


# 53. `persistent_workers`

When:

```python
num_workers > 0
```

you can keep worker processes alive across epochs with:

```python
persistent_workers=True
```

This may reduce worker startup overhead.

Do not enable it when:

```python
num_workers=0
```


# 54. Practical DataLoader Builder


In [ ]:
def make_loader(
    dataset,
    batch_size,
    shuffle,
    device,
    num_workers=0
):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(
            device.type == "cuda"
        ),
        persistent_workers=(
            num_workers > 0
        )
    )

example_loader = make_loader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    device=device,
    num_workers=0
)

print(
    example_loader
)


# 55. Checkpoint Resuming

Saving the best model is useful for evaluation.

But resuming interrupted training requires more information.

A resume checkpoint should usually contain:

- Model state
- Optimizer state
- Scheduler state
- Scaler state if using AMP
- Current epoch
- Best validation metric
- Experiment configuration


# 56. Why Optimizer State Matters

Optimizers such as Adam maintain internal state.

If you reload only model weights but create a fresh optimizer, you do **not** resume from the exact same optimization state.

For faithful continuation:

```python
optimizer.load_state_dict(...)
```


# 57. Why Scheduler State Matters

Schedulers also maintain progress.

If you trained for:

$$
20
$$

epochs and then restart the scheduler from epoch 0, the learning-rate schedule may be wrong.

Save and restore:

```python
scheduler.state_dict()
```


# 58. Why GradScaler State Matters

AMP's `GradScaler` dynamically changes its scaling factor.

To resume mixed-precision training faithfully, save:

```python
scaler.state_dict()
```


# 59. Creating a Full Resume Checkpoint


In [ ]:
resume_model = SmallMLP().to(
    device
)

resume_optimizer = torch.optim.AdamW(
    resume_model.parameters(),
    lr=1e-3
)

resume_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        resume_optimizer,
        T_max=20
    )
)

resume_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)

checkpoint = {
    "epoch":
        5,

    "model_state_dict":
        resume_model.state_dict(),

    "optimizer_state_dict":
        resume_optimizer.state_dict(),

    "scheduler_state_dict":
        resume_scheduler.state_dict(),

    "scaler_state_dict":
        resume_scaler.state_dict(),

    "best_val_loss":
        0.42,

    "config":
        {
            "learning_rate": 1e-3,
            "epochs": 20
        }
}

checkpoint_path = Path(
    "training_resume_checkpoint.pth"
)

torch.save(
    checkpoint,
    checkpoint_path
)

print(
    "Saved:",
    checkpoint_path
)


# 60. Loading a Resume Checkpoint


In [ ]:
loaded_model = SmallMLP().to(
    device
)

loaded_optimizer = torch.optim.AdamW(
    loaded_model.parameters(),
    lr=1e-3
)

loaded_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        loaded_optimizer,
        T_max=20
    )
)

loaded_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=amp_enabled
)

loaded_checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

loaded_model.load_state_dict(
    loaded_checkpoint[
        "model_state_dict"
    ]
)

loaded_optimizer.load_state_dict(
    loaded_checkpoint[
        "optimizer_state_dict"
    ]
)

loaded_scheduler.load_state_dict(
    loaded_checkpoint[
        "scheduler_state_dict"
    ]
)

loaded_scaler.load_state_dict(
    loaded_checkpoint[
        "scaler_state_dict"
    ]
)

start_epoch = (
    loaded_checkpoint[
        "epoch"
    ]
    + 1
)

best_val_loss = (
    loaded_checkpoint[
        "best_val_loss"
    ]
)

print(
    "Resume from epoch:",
    start_epoch
)

print(
    "Best val loss:",
    best_val_loss
)


# 61. Checkpoint Compatibility

To resume successfully, these must be compatible:

- Architecture
- Optimizer parameter structure
- Scheduler type
- Number/order of parameter groups
- AMP scaler usage

Changing the model architecture may make the old checkpoint incompatible.


# 62. Training-Loop Instrumentation

Instrumentation means collecting information about training while it runs.

Useful signals include:

- Train loss
- Validation loss
- Accuracy
- Learning rate
- Gradient norm
- Epoch time
- Batch time
- GPU memory
- Best checkpoint


# 63. Measuring Epoch Time


In [ ]:
start_time = time.perf_counter()

_ = train_epoch_basic(
    step_model,
    train_loader,
    criterion,
    step_optimizer,
    device
)

elapsed = (
    time.perf_counter()
    - start_time
)

print(
    f"Epoch time: {elapsed:.3f} seconds"
)


# 64. Tracking Gradient Norm

Gradient norm can reveal instability.

A helper:


In [ ]:
def total_gradient_norm(
    model
):
    squared_sum = 0.0

    for parameter in (
        model.parameters()
    ):
        if parameter.grad is not None:
            squared_sum += (
                parameter.grad.norm().item()
                ** 2
            )

    return math.sqrt(
        squared_sum
    )


# 65. Tracking the Learning Rate


In [ ]:
def current_learning_rates(
    optimizer
):
    return [
        group["lr"]
        for group in optimizer.param_groups
    ]

print(
    current_learning_rates(
        step_optimizer
    )
)


# 66. GPU Memory Instrumentation

When CUDA is available, PyTorch can report memory statistics.

Useful functions include:

```python
torch.cuda.memory_allocated()
torch.cuda.max_memory_allocated()
```

These help diagnose out-of-memory pressure.


In [ ]:
if torch.cuda.is_available():
    print(
        "Allocated MB:",
        torch.cuda.memory_allocated()
        / 1024**2
    )

    print(
        "Max allocated MB:",
        torch.cuda.max_memory_allocated()
        / 1024**2
    )

else:
    print(
        "CUDA memory metrics unavailable on CPU."
    )


# 67. Resetting Peak GPU Memory Stats

Before measuring one experiment or epoch:

```python
torch.cuda.reset_peak_memory_stats()
```

Then inspect peak memory afterward.


In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

    print(
        "Peak-memory counter reset."
    )


# 68. A Simple Metric History Dictionary


In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "learning_rate": [],
    "epoch_time": []
}

print(
    history.keys()
)


# 69. Instrumentation Should Not Dominate Training

Logging every scalar every microsecond can slow training.

Choose a reasonable frequency.

Examples:

- Loss every N batches
- Metrics every epoch
- Images occasionally
- Full confusion matrix only after validation


# 70. Practical Performance Optimization — Measure First

Do not optimize blindly.

First identify the bottleneck:

- Data loading?
- GPU compute?
- CPU preprocessing?
- Device transfer?
- Model architecture?
- Batch size?

Optimization should target the actual bottleneck.


# 71. Increase Batch Size Until It Stops Helping

A larger batch can improve hardware utilization.

But it also:

- Uses more memory
- Changes optimization behavior
- May require learning-rate tuning

The best batch size is not simply the largest one that fits.


# 72. Avoid Unnecessary CPU-GPU Synchronization

Calling:

```python
loss.item()
```

forces a scalar result to become available on the CPU.

Logging it once per batch is usually fine.

But excessive `.item()` calls inside performance-critical code can add synchronization overhead.


# 73. Avoid Moving Tensors Back and Forth Repeatedly

Bad pattern:

```python
GPU -> CPU -> GPU -> CPU
```

Keep tensors on the accelerator for as long as the computation requires them.

Move final predictions to CPU only when needed for:

- Storage
- NumPy
- Plotting
- CPU metric libraries


# 74. Avoid Keeping Computation Graphs Accidentally

For logging:

```python
loss.item()
```

For stored predictions:

```python
tensor.detach().cpu()
```

Do not append graph-connected tensors from every batch unless you actually need the graph.


# 75. `torch.compile` — Optional Advanced Optimization

Modern PyTorch can compile models/functions for potential speedups:

```python
compiled_model = torch.compile(model)
```

Benefits depend on:

- Model
- Hardware
- Input shapes
- PyTorch version

Compilation has startup overhead, so benchmark end-to-end.


In [ ]:
compile_available = hasattr(
    torch,
    "compile"
)

print(
    "torch.compile available:",
    compile_available
)


# 76. Using `torch.compile` Safely

A simple optional pattern is:

```python
if hasattr(torch, "compile"):
    model = torch.compile(model)
```

For teaching and debugging, begin with the eager model first.

Compile only after correctness is established.


# 77. Why Debug Before Optimizing Performance?

A faster wrong model is still wrong.

The recommended order is:

$$
\boxed{
Correctness
\rightarrow
Reproducibility
\rightarrow
Profiling
\rightarrow
Optimization
}
$$


# 78. A More Complete Advanced Training Loop

We will combine:

- AMP
- Gradient accumulation
- Gradient clipping
- Scheduler
- Validation
- Instrumentation


In [ ]:
def evaluate_accuracy(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    loss_sum = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for inputs, targets in loader:
            inputs = inputs.to(
                device,
                non_blocking=(
                    device.type == "cuda"
                )
            )

            targets = targets.to(
                device,
                non_blocking=(
                    device.type == "cuda"
                )
            )

            logits = model(
                inputs
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = targets.size(0)

            loss_sum += (
                loss.item()
                * batch_size
            )

            correct += (
                logits.argmax(
                    dim=1
                )
                == targets
            ).sum().item()

            total += batch_size

    return (
        loss_sum / total,
        correct / total
    )


In [ ]:
def fit_advanced(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs,
    accumulation_steps=1,
    max_grad_norm=None,
    use_amp=True
):
    use_amp = (
        use_amp
        and device.type == "cuda"
    )

    dtype = (
        torch.float16
        if device.type == "cuda"
        else torch.bfloat16
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": [],
        "epoch_time": []
    }

    best_val_loss = float(
        "inf"
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    for epoch in range(
        epochs
    ):
        epoch_start = (
            time.perf_counter()
        )

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        train_loss_sum = 0.0
        train_total = 0

        for batch_index, (
            inputs,
            targets
        ) in enumerate(
            train_loader
        ):
            inputs = inputs.to(
                device,
                non_blocking=(
                    device.type
                    == "cuda"
                )
            )

            targets = targets.to(
                device,
                non_blocking=(
                    device.type
                    == "cuda"
                )
            )

            with torch.autocast(
                device_type=device.type,
                dtype=dtype,
                enabled=use_amp
            ):
                logits = model(
                    inputs
                )

                raw_loss = criterion(
                    logits,
                    targets
                )

                loss = (
                    raw_loss
                    / accumulation_steps
                )

            scaler.scale(
                loss
            ).backward()

            should_step = (
                (batch_index + 1)
                % accumulation_steps
                == 0
            )

            is_last_batch = (
                batch_index + 1
                == len(train_loader)
            )

            if should_step or is_last_batch:
                if (
                    max_grad_norm
                    is not None
                ):
                    scaler.unscale_(
                        optimizer
                    )

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=max_grad_norm
                    )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

            batch_size = (
                targets.size(0)
            )

            train_loss_sum += (
                raw_loss.item()
                * batch_size
            )

            train_total += (
                batch_size
            )

        train_loss = (
            train_loss_sum
            / train_total
        )

        val_loss, val_accuracy = (
            evaluate_accuracy(
                model,
                val_loader,
                criterion,
                device
            )
        )

        lr = optimizer.param_groups[
            0
        ]["lr"]

        epoch_time = (
            time.perf_counter()
            - epoch_start
        )

        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "val_accuracy"
        ].append(
            val_accuracy
        )

        history[
            "learning_rate"
        ].append(
            lr
        )

        history[
            "epoch_time"
        ].append(
            epoch_time
        )

        if val_loss < best_val_loss:
            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

        if isinstance(
            scheduler,
            torch.optim.lr_scheduler.ReduceLROnPlateau
        ):
            scheduler.step(
                val_loss
            )

        elif scheduler is not None:
            scheduler.step()

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train {train_loss:.4f} | "
            f"Val {val_loss:.4f} | "
            f"Acc {val_accuracy:.3f} | "
            f"LR {lr:.6f} | "
            f"Time {epoch_time:.2f}s"
        )

    model.load_state_dict(
        best_state
    )

    return (
        history,
        scaler
    )


# 79. Running the Advanced Loop


In [ ]:
torch.manual_seed(42)

final_model = SmallMLP().to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=5,
        eta_min=1e-5
    )
)

history, scaler = fit_advanced(
    final_model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs=5,
    accumulation_steps=2,
    max_grad_norm=1.0,
    use_amp=True
)


# 80. Important Scheduler Interaction With Gradient Accumulation

If a scheduler is designed to step per **optimizer update**, gradient accumulation changes how often optimizer updates happen.

Example:

Without accumulation:

$$
100
$$

batches:

$$
100
$$

optimizer steps.

With:

$$
accumulation\_steps=4
$$

you get only about:

$$
25
$$

optimizer steps.

A step-based scheduler should account for this.


# 81. Scheduler State and Parameter Groups

If an optimizer has several parameter groups, the scheduler may update all group learning rates.

Example from transfer learning:

$$
\eta_{backbone}=10^{-4}
$$

$$
\eta_{head}=10^{-3}
$$

A multiplicative scheduler can preserve the relative ratio while changing both.


# 82. Common Mistake — Calling `ReduceLROnPlateau.step()` Without a Metric

Wrong:

```python
scheduler.step()
```

for `ReduceLROnPlateau`.

It needs the monitored metric:

```python
scheduler.step(
    val_loss
)
```


# 83. Common Mistake — Scheduler Steps at the Wrong Frequency

If a scheduler is intended to step once per epoch but you call it every batch, the learning rate can decay far too quickly.

Always know whether it is:

- Epoch-based
- Step-based
- Metric-based


# 84. Common Mistake — Applying Softmax Inside AMP Training Before CrossEntropy

Mixed precision does not change loss rules.

For `CrossEntropyLoss`:

```python
loss = criterion(
    logits,
    targets
)
```

Use raw logits.


# 85. Common Mistake — Clipping Scaled Gradients

With AMP:

1. `backward()` on scaled loss
2. `scaler.unscale_(optimizer)`
3. Clip
4. `scaler.step(optimizer)`

Clipping before unscaling uses the wrong gradient scale.


# 86. Common Mistake — Forgetting to Divide Loss During Accumulation

Without dividing by `accumulation_steps`, accumulated gradients become approximately larger.

That changes the effective update magnitude.

Use:

```python
loss = raw_loss / accumulation_steps
```

when approximating a larger average batch.


# 87. Common Mistake — Calling `optimizer.step()` Every Micro-Batch

If you call `step()` after every micro-batch, you are not accumulating gradients.

You are simply training with the smaller batch size.


# 88. Common Mistake — Forgetting the Final Partial Accumulation

Suppose there are 10 mini-batches and:

$$
accumulation\_steps=4
$$

Updates occur after:

- Batch 4
- Batch 8
- Final batch 10

If you only step when divisible by 4, gradients from batches 9–10 are lost.

Handle the final partial group explicitly.


# 89. Common Mistake — Using `model.train()` During Inference

Inference should usually use:

```python
model.eval()
```

Otherwise:

- Dropout stays random
- BatchNorm uses training behavior


# 90. Common Mistake — Saving Only Model Weights When You Need to Resume

Model weights are enough for inference.

They are not enough for exact training continuation.

For resuming, also save:

- Optimizer
- Scheduler
- GradScaler
- Epoch


# 91. Common Mistake — Optimizing Performance Before Correctness

Do not begin with:

- Mixed precision
- Many workers
- Compilation
- Complex schedulers

until a simple baseline is correct.

A strong order is:

$$
\boxed{
Correct
\rightarrow
Reproducible
\rightarrow
Stable
\rightarrow
Fast
}
$$


# 92. Practical Performance Checklist

When training is slow:

1. Is GPU actually being used?
2. Is GPU utilization low because data loading is slow?
3. Can batch size increase?
4. Can AMP help?
5. Is `pin_memory=True` useful?
6. Would more DataLoader workers help?
7. Are there excessive CPU-GPU transfers?
8. Are you synchronizing too often?
9. Can inference use larger batches?
10. Can `torch.compile` help after correctness is established?


# 93. Advanced Training Checklist

Before a long experiment, verify:

1. Optimizer learning rate
2. Scheduler type
3. Scheduler step frequency
4. AMP enabled only where supported
5. GradScaler state
6. Gradient accumulation factor
7. Effective batch size
8. Gradient clipping order
9. Best-checkpoint logic
10. Resume checkpoint completeness
11. Validation uses inference/eval mode
12. Instrumentation records learning rate and timing


# 94. Practice Exercises

Try these before looking at the solutions.

## Exercise 1

Create an SGD optimizer with:

$$
lr=0.1
$$

and `StepLR` with:

$$
step\_size=3,\ gamma=0.5
$$

## Exercise 2

Create a `ReduceLROnPlateau` scheduler that reduces LR by half after 2 non-improving epochs.

## Exercise 3

Create a `CosineAnnealingLR` schedule for 20 epochs.

## Exercise 4

Write a simple 5-step linear warmup using `LambdaLR`.

## Exercise 5

Write one mixed-precision training step using `torch.autocast`.

## Exercise 6

Add `GradScaler`.

## Exercise 7

Accumulate gradients over 4 micro-batches.

## Exercise 8

Clip gradient norm to:

$$
1.0
$$

after unscaling.

## Exercise 9

Write an inference loop using `torch.inference_mode()`.

## Exercise 10

Save a resume checkpoint containing model, optimizer, scheduler, scaler, and epoch.


# 95. Conceptual Challenges

## Challenge 1

Why can a decreasing learning rate help later training?

## Challenge 2

How does `ReduceLROnPlateau` differ from `StepLR`?

## Challenge 3

Why can warmup stabilize early training?

## Challenge 4

Why can mixed precision reduce GPU memory use?

## Challenge 5

Why is gradient scaling useful with `float16`?

## Challenge 6

Why must gradients be unscaled before clipping?

## Challenge 7

How does gradient accumulation increase effective batch size?

## Challenge 8

Why can BatchNorm make gradient accumulation differ from a true large batch?

## Challenge 9

Why is `torch.inference_mode()` useful?

## Challenge 10

Why must optimizer and scheduler states be saved when resuming training?


# 96. Exercise Solutions


In [ ]:
# Exercise 1
exercise_model = SmallMLP()

exercise_optimizer = torch.optim.SGD(
    exercise_model.parameters(),
    lr=0.1
)

exercise_step_scheduler = (
    torch.optim.lr_scheduler.StepLR(
        exercise_optimizer,
        step_size=3,
        gamma=0.5
    )
)

# Exercise 2
exercise_plateau = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        exercise_optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )
)

# Exercise 3
exercise_cosine = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        exercise_optimizer,
        T_max=20
    )
)

print(
    "Exercises 1-3 complete."
)


In [ ]:
# Exercise 4
warmup_steps = 5

exercise_warmup = (
    torch.optim.lr_scheduler.LambdaLR(
        exercise_optimizer,
        lr_lambda=lambda step: min(
            1.0,
            (step + 1)
            / warmup_steps
        )
    )
)

print(
    "Exercise 4 complete."
)


In [ ]:
# Exercise 5 and 6
exercise_model = SmallMLP().to(
    device
)

exercise_optimizer = torch.optim.AdamW(
    exercise_model.parameters(),
    lr=1e-3
)

exercise_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        device.type == "cuda"
    )
)

inputs, targets = next(
    iter(train_loader)
)

inputs = inputs.to(
    device
)

targets = targets.to(
    device
)

exercise_optimizer.zero_grad(
    set_to_none=True
)

with torch.autocast(
    device_type=device.type,
    dtype=(
        torch.float16
        if device.type == "cuda"
        else torch.bfloat16
    ),
    enabled=(
        device.type == "cuda"
    )
):
    logits = exercise_model(
        inputs
    )

    loss = nn.CrossEntropyLoss()(
        logits,
        targets
    )

exercise_scaler.scale(
    loss
).backward()

exercise_scaler.step(
    exercise_optimizer
)

exercise_scaler.update()

print(
    "Exercises 5-6 loss:",
    loss.item()
)


In [ ]:
# Exercise 7 and 8
exercise_model = SmallMLP().to(
    device
)

exercise_optimizer = torch.optim.AdamW(
    exercise_model.parameters(),
    lr=1e-3
)

accumulation_steps = 4

exercise_optimizer.zero_grad(
    set_to_none=True
)

for batch_index, (
    inputs,
    targets
) in enumerate(
    train_loader
):
    inputs = inputs.to(
        device
    )

    targets = targets.to(
        device
    )

    loss = nn.CrossEntropyLoss()(
        exercise_model(
            inputs
        ),
        targets
    )

    (
        loss
        / accumulation_steps
    ).backward()

    should_step = (
        (batch_index + 1)
        % accumulation_steps
        == 0
    )

    is_last = (
        batch_index + 1
        == len(train_loader)
    )

    if should_step or is_last:
        torch.nn.utils.clip_grad_norm_(
            exercise_model.parameters(),
            max_norm=1.0
        )

        exercise_optimizer.step()

        exercise_optimizer.zero_grad(
            set_to_none=True
        )

print(
    "Exercises 7-8 complete."
)


In [ ]:
# Exercise 9
exercise_model.eval()

predictions = []

with torch.inference_mode():
    for inputs, _ in val_loader:
        inputs = inputs.to(
            device
        )

        logits = exercise_model(
            inputs
        )

        predictions.append(
            logits.argmax(
                dim=1
            ).cpu()
        )

predictions = torch.cat(
    predictions
)

print(
    "Exercise 9 predictions:",
    predictions.shape
)


In [ ]:
# Exercise 10
exercise_scheduler = (
    torch.optim.lr_scheduler.StepLR(
        exercise_optimizer,
        step_size=3,
        gamma=0.5
    )
)

exercise_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        device.type == "cuda"
    )
)

exercise_checkpoint = {
    "epoch":
        3,

    "model_state_dict":
        exercise_model.state_dict(),

    "optimizer_state_dict":
        exercise_optimizer.state_dict(),

    "scheduler_state_dict":
        exercise_scheduler.state_dict(),

    "scaler_state_dict":
        exercise_scaler.state_dict()
}

torch.save(
    exercise_checkpoint,
    "exercise_resume_checkpoint.pth"
)

print(
    "Exercise 10 saved."
)


# 97. Key Takeaways

In this notebook, we learned:

- Why learning rates can change during training
- `StepLR`
- `ReduceLROnPlateau`
- Cosine annealing
- Warmup intuition
- Epoch-based vs step-based schedulers
- Mixed-precision training
- `torch.autocast`
- Gradient scaling
- `torch.amp.GradScaler`
- Gradient accumulation
- Effective batch size
- BatchNorm nuance during accumulation
- Gradient clipping with AMP
- `optimizer.zero_grad(set_to_none=True)`
- Efficient inference
- `torch.inference_mode()`
- Batched inference
- `pin_memory`
- `non_blocking=True`
- `num_workers`
- `persistent_workers`
- Full resume checkpoints
- Optimizer state
- Scheduler state
- Scaler state
- Training-loop instrumentation
- Epoch timing
- Gradient norms
- GPU-memory monitoring
- `torch.compile` intuition
- Practical performance optimization

The advanced update order to remember is:

$$
\boxed{
zero\_grad
\rightarrow
autocast
\rightarrow
loss
\rightarrow
scaled\ backward
\rightarrow
unscale
\rightarrow
clip
\rightarrow
optimizer\ step
\rightarrow
scaler\ update
}
$$

With accumulation:

$$
\boxed{
\text{Several Micro-Batches}
\rightarrow
\text{One Optimizer Step}
}
$$

And the engineering principle is:

$$
\boxed{
Correctness
\rightarrow
Reproducibility
\rightarrow
Stability
\rightarrow
Performance
}
$$


# 98. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What does a learning-rate scheduler do?
2. How does `StepLR` work?
3. What metric does `ReduceLROnPlateau` usually monitor?
4. Why does `ReduceLROnPlateau.step()` need a metric?
5. What is cosine annealing?
6. What is warmup?
7. Why can scheduler step frequency matter?
8. What is mixed-precision training?
9. What does `torch.autocast` do?
10. Why is gradient scaling useful?
11. What does `GradScaler.step()` do?
12. Why must gradients be unscaled before clipping?
13. What is gradient accumulation?
14. How do you calculate effective batch size?
15. Why divide the loss by `accumulation_steps`?
16. Why can BatchNorm behave differently under gradient accumulation?
17. What does `set_to_none=True` do?
18. How is `torch.inference_mode()` different from ordinary training?
19. Why can inference batch size be larger than training batch size?
20. When can `pin_memory=True` help?
21. What does `non_blocking=True` try to improve?
22. Why save optimizer state when resuming?
23. Why save scheduler state?
24. Why save GradScaler state?
25. What training metrics are useful to instrument?
26. Why should performance optimization come after correctness?
27. When might `torch.compile` help?
28. Why can too many DataLoader workers hurt?
29. Why should test performance not guide scheduler tuning?
30. What is the correct high-level order for AMP + clipping + optimizer step?


# Next Notebook

# 21 — Evaluation Metrics and Model Analysis

In the next notebook, we will study:

- Why accuracy is not enough
- Confusion matrices
- True positives, false positives, true negatives, false negatives
- Precision
- Recall / sensitivity
- Specificity
- F1 score
- Class imbalance
- ROC curves
- AUROC
- Precision-recall curves
- AUPRC
- Threshold selection
- Multi-class metrics
- Per-class analysis
- Calibration intuition
- Error analysis
- Choosing metrics for medical imaging
